# 06. Street commerce

This notebook builds the street commerce layer: food sold from stalls and by
itinerant vendors, which the directory of economic units does not carry because
it lists establishments with an address.

The source is the audit of the urban environment collected alongside the
population and housing census. Enumerators walked every block face in every
urban locality and recorded the features present on it, and two of those
features are semi fixed stalls and itinerant vendors.

The audit is published at two levels. One file holds a row for every block
face, with a presence code for each feature. The other holds a row for every
block, with each feature summarised as the number of that block's sides on
which it was present. This notebook reads the block level file, which is the
unit used throughout, and keeps the number of sides alongside the presence of
any.

Each feature arrives as three values: a classification code for the block, the
number of sides on which the feature was present, and the number on which it
was absent. The two counts sum to the number of sides of the block wherever
every face of it was covered. Presence is taken from the count of sides, the
correspondence between those counts and the classification code is printed, and
the codes above three, which mark blocks whose faces were not all covered, are
flagged.

The audit records that a stall was present. It does not record what the stall
sold, and stalls sell food, clothing, telephone accessories and household
goods. The layer is the presence of street commerce, and an upper bound on the
reach of food sold in the street. Semi fixed stalls and itinerant vendors are
reported separately, because the share of each that sells food is not the same,
and the block level file gives no way to count the sides carrying either kind.

Street commerce occupies the public way, so it has no address and no
coordinates. A block with stalls on its own sides has street food at its
doorstep, and for any other block the quantity of interest is the distance to
the nearest block that has it. The layer is an attribute of the block, and it
becomes a set of destinations when travel times are computed.

Inputs are the audit, `Inputs/CLEU2020/Censo2020_CEU_eum_csv.zip`, and the
analytical frame, `Results/04_blocks.parquet` and `Results/04_tracts.parquet`.
The frame is the set of urban census blocks lying inside the boundaries of the
92 metropolitan areas, each carrying its population, its metropolitan area and
the deprivation grade of its tract. The audit joins to it on the block key, the
state, municipality, locality, tract and block numbers concatenated.

Output is `Results/06_street_commerce.parquet`, one row per block in the frame,
carrying the presence of each kind of street commerce, the number of block
sides on which it was recorded, the share of the sides of the block that
represents, and whether the block was covered in full.

Runtime is a few minutes, most of it reading the audit.

## 1. Setup

In [ ]:
import sys, os, io, re, zipfile, importlib

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = os.environ.get('MXO_BASE', '/content/drive/MyDrive/Mexico_Obesity')
os.environ['MXO_BASE'] = PROJECT
sys.path.insert(0, f'{PROJECT}/Code')
import mxo_utils as mx
importlib.reload(mx)

assert mx.BASE == PROJECT, f'the module resolves to {mx.BASE}'

import numpy as np
import pandas as pd

rep = mx.Report('06_street_commerce').start()
mx.flow_reset()

INPUTS = mx.DIRS['inputs']
RESULTS = mx.DIRS['results']

BLOCKS = pd.read_parquet(os.path.join(RESULTS, '04_blocks.parquet'),
                         columns=['CVEGEO_MZA', 'CVEGEO', 'CVE_ENT', 'slug',
                                  'macro_region', 'POBTOT'])
TRACTS = pd.read_parquet(os.path.join(RESULTS, '04_tracts.parquet'),
                         columns=['CVEGEO', 'slug', 'grs_grade', 'veh_share',
                                  'POBTOT', 'block_weightable'])

print(f'{len(BLOCKS):,} blocks in the frame, '
      f'{BLOCKS.POBTOT.sum():,.0f} people, '
      f'{BLOCKS.CVEGEO.nunique():,} tracts')

## 2. The audit archive

The archive holds one member per level of aggregation. Both are listed with
their uncompressed size, and the block level member is selected by name. A
member of a compressed archive cannot be read a second time from its beginning
once reading has started, so the member is held in memory before being decoded
and decoding is attempted in more than one encoding.

In [ ]:
ARCH = os.path.join(INPUTS, 'CLEU2020', 'Censo2020_CEU_eum_csv.zip')
assert os.path.exists(ARCH), ARCH

with zipfile.ZipFile(ARCH) as zf:
    members = [i for i in zf.infolist() if not i.is_dir()]
    print('members of the archive:\n')
    for i in sorted(members, key=lambda x: -x.file_size):
        print(f'   {i.file_size / 1e6:9.1f} MB  {i.filename}')

    BLOCK_MEMBER = re.compile(r'MANZANA', re.I)
    chosen = [i.filename for i in members
              if BLOCK_MEMBER.search(os.path.basename(i.filename))
              and i.filename.lower().endswith('.csv')]
    assert len(chosen) == 1, f'expected one block level member, found {chosen}'
    print(f'\nreading {chosen[0]}')

    with mx.stage('read the audit'):
        raw = zf.read(chosen[0])

KEYS = ['ENT', 'MUN', 'LOC', 'AGEB', 'MZA']
FEATURES = {'semi': ('PUESSEMI_C', 'C_PSEMI_N', 'S_PSEMI_N'),
            'ambu': ('PUESAMBU_C', 'C_PAMBU_N', 'S_PAMBU_N')}
EXTRA = ['ID_MZA', 'PERSONAS', 'VIVPARHAB', 'TOTLADO', 'TOTVIAL',
         'TOTRASGO', 'TAMLOC']
USE = KEYS + EXTRA + [col for cols in FEATURES.values() for col in cols]

for enc in ('utf-8', 'latin1'):
    try:
        head = pd.read_csv(io.BytesIO(raw), nrows=0, encoding=enc)
        encoding = enc
        break
    except UnicodeDecodeError:
        continue

missing = [col for col in USE if col not in head.columns]
assert not missing, f'the audit lacks {missing} among its {len(head.columns)} variables'

A = pd.read_csv(io.BytesIO(raw), usecols=USE, dtype=str, encoding=encoding)
del raw

print(f'\n{len(A):,} blocks in the audit, {len(head.columns)} variables '
      f'published, {len(USE)} read, encoding {encoding}')
print('The audit covers every urban locality in the country. The frame covers '
      'the 92 metropolitan areas.')

## 3. What the published values are

Each feature carries three variables: a classification code for the block, the
number of block faces on which the feature was present, and the number on which
it was absent. Their sum is the number of faces the audit recorded for that
feature on that block, and shares below are taken over it.

The file also publishes three counts of the block itself: its sides, the streets
bounding it, and the other features bounding it, such as a watercourse or a
railway. The recorded faces are compared with each, which identifies what the
faces are faces of.

In [ ]:
TOTALS = ['TOTLADO', 'TOTVIAL', 'TOTRASGO']
NUM = TOTALS + [col for cols in FEATURES.values() for col in cols[1:]]
for col in NUM + ['PERSONAS', 'VIVPARHAB']:
    A[col] = pd.to_numeric(A[col], errors='coerce')

for short, (code_col, with_col, without_col) in FEATURES.items():
    print(f'--- {short}: {code_col}, {with_col}, {without_col} ---\n')
    A[f'faces_{short}'] = A[with_col].fillna(0).astype(int)
    A[f'nofaces_{short}'] = A[without_col].fillna(0).astype(int)
    A[f'recorded_{short}'] = A[f'faces_{short}'] + A[f'nofaces_{short}']
    tab = (A.groupby(code_col)
             .agg(blocks=(with_col, 'size'),
                  faces_with_min=(f'faces_{short}', 'min'),
                  faces_with_max=(f'faces_{short}', 'max'),
                  faces_without_max=(f'nofaces_{short}', 'max'),
                  faces_recorded_min=(f'recorded_{short}', 'min'),
                  faces_recorded_max=(f'recorded_{short}', 'max'))
             .reset_index())
    tab['share_of_blocks'] = (tab['blocks'] / len(A)).round(4)
    print(tab.to_string(index=False))
    rep.add_table(f'published values for {short}', tab, max_rows=20)
    print()

print('--- recorded faces against the published counts of the block ---\n')
rows = []
for short in FEATURES:
    rec = A[f'recorded_{short}']
    for label, total in [('sides', A.TOTLADO),
                         ('bounding streets', A.TOTVIAL),
                         ('other bounding features', A.TOTRASGO),
                         ('streets and other features', A.TOTVIAL + A.TOTRASGO),
                         ('sides and other features', A.TOTLADO + A.TOTRASGO)]:
        rows.append({'feature': short, 'compared with': label,
                     'blocks equal': int((rec == total).sum()),
                     'share equal': round(float((rec == total).mean()), 4),
                     'recorded greater': int((rec > total).sum()),
                     'recorded smaller': int((rec < total).sum())})
DEN = pd.DataFrame(rows)
print(DEN.to_string(index=False))
rep.add_table('recorded faces against the published block counts', DEN,
              max_rows=20)

print('\n--- the published counts themselves ---\n')
print(A[TOTALS + [f'recorded_{s}' for s in FEATURES]]
       .describe(percentiles=[0.05, 0.5, 0.95]).round(2).to_string())

The classification code and the recorded faces carry the same information. A
block with no recorded face for a feature takes one of the two highest codes. A
block with at least one takes one of the three lowest, and there the code
follows from the counts: present on every recorded face, present on some, or
present on none. The table sets out that correspondence as it appears in the
file. A block with no recorded face is one the audit did not reach, where an
absence of stalls is not evidence that there are none, so it is flagged.

In [ ]:
rows = []
for short, (code_col, with_col, without_col) in FEATURES.items():
    rec = A[f'recorded_{short}'] > 0
    face = A[f'faces_{short}']
    for label, mask in [('every recorded face', rec & (face == A[f'recorded_{short}'])),
                        ('some recorded faces', rec & (face > 0)
                                                & (face < A[f'recorded_{short}'])),
                        ('no recorded face', rec & (face == 0)),
                        ('no face recorded at all', ~rec)]:
        if int(mask.sum()):
            rows.append({'feature': short, 'present on': label,
                         'blocks': int(mask.sum()),
                         'share': round(float(mask.mean()), 4),
                         'codes present': ', '.join(
                             sorted(A.loc[mask, code_col].dropna().unique()))})
CODES = pd.DataFrame(rows)
print(CODES.to_string(index=False))
rep.add_table('codes against the recorded faces', CODES, max_rows=20)

## 4. Deriving the layer

A block carries street commerce of a given kind when at least one of its
recorded faces carried it. The count of faces is kept alongside the flag, as a
share of the faces recorded for that block. The two kinds stay separate,
because a block can carry one on one face and the other on another and the
block level file publishes only each count.

In [ ]:
A['CVEGEO_MZA'] = mx.build_cvegeo_mza(A['ENT'], A['MUN'], A['LOC'],
                                      A['AGEB'], A['MZA']).values
A['CVEGEO'] = mx.build_cvegeo(A['ENT'], A['MUN'], A['LOC'], A['AGEB']).values
A['sides_total'] = A['TOTLADO'].fillna(0).astype(int)

for short in FEATURES:
    A[f'street_{short}'] = A[f'faces_{short}'] > 0
    A[f'share_{short}'] = (A[f'faces_{short}']
                           / A[f'recorded_{short}'].replace(0, np.nan)).round(4)

A['street_any'] = A['street_semi'] | A['street_ambu']
A['street_both'] = A['street_semi'] & A['street_ambu']
A['faces_recorded'] = A[['recorded_semi', 'recorded_ambu']].min(axis=1)
A['street_recorded'] = A['faces_recorded'] > 0

print('nationally, across every block in the audit:\n')
summary = pd.DataFrame([
    {'measure': m, 'blocks': int(A[col].sum()),
     'share_of_blocks': round(float(A[col].mean()), 4)}
    for m, col in [('with at least one recorded face', 'street_recorded'),
                   ('semi fixed stalls on a recorded face', 'street_semi'),
                   ('itinerant vendors on a recorded face', 'street_ambu'),
                   ('either kind', 'street_any'),
                   ('both kinds', 'street_both')]])
print(summary.to_string(index=False))
rep.add_table('street commerce across the national audit', summary, max_rows=10)

print('\nfaces carrying each kind, among the blocks that carry it:\n')
for short, label in [('semi', 'semi fixed stalls'),
                     ('ambu', 'itinerant vendors')]:
    s = A.loc[A[f'street_{short}'], f'share_{short}'].dropna()
    print(f'   {label:24s} median {s.median():.3f}, '
          f'mean {s.mean():.3f}, on {len(s):,} blocks')

## 5. Joining to the analytical frame

The audit covers every urban locality in the country and the frame covers the
92 metropolitan areas, so the join restricts the audit to the frame. The two
sources delimit the urban universe in their own ways, so blocks in the frame
that are absent from the audit are counted and listed by area.

In [ ]:
cols = ['CVEGEO_MZA', 'sides_total', 'faces_semi', 'faces_ambu',
        'faces_recorded', 'share_semi', 'share_ambu',
        'street_semi', 'street_ambu', 'street_any', 'street_both',
        'street_recorded', 'PERSONAS', 'VIVPARHAB']

B = BLOCKS.merge(A[cols].drop_duplicates('CVEGEO_MZA'),
                 on='CVEGEO_MZA', how='left', indicator='_src')
matched = B['_src'].eq('both')

print(f'blocks in the frame                   {len(B):,}')
print(f'blocks present in the audit           {int(matched.sum()):,} '
      f'({matched.mean():.2%})')
print(f'blocks absent from the audit          {int((~matched).sum()):,}, '
      f'{int(B.loc[~matched, "POBTOT"].sum()):,} people')

if int((~matched).sum()):
    print('\nareas contributing the most blocks absent from the audit:')
    absent = (B.loc[~matched].groupby('slug')
                .agg(blocks=('CVEGEO_MZA', 'size'), population=('POBTOT', 'sum'))
                .sort_values('blocks', ascending=False))
    print(absent.head(12).to_string())
    rep.add_table('blocks absent from the audit, by area',
                  absent.reset_index(), max_rows=92)

B = B.drop(columns='_src')
for col in ['street_semi', 'street_ambu', 'street_any', 'street_both',
            'street_recorded']:
    B[col] = B[col].fillna(False).astype(bool)
for col in ['sides_total', 'faces_semi', 'faces_ambu', 'faces_recorded']:
    B[col] = B[col].fillna(0).astype(int)
B['in_audit'] = matched.values

# A block the audit never reached, and a block it reached without recording any
# face, are both blocks whose street commerce is unknown. They stay in the
# frame as origins and are not available as destinations.
B['street_known'] = B['in_audit'] & B['street_recorded']

mx.flow('blocks in the frame', len(BLOCKS), 'from the census join')
mx.flow('blocks matched to the audit', int(matched.sum()),
        'absent blocks flagged, not dropped')
mx.flow('blocks with street commerce known', int(B.street_known.sum()),
        'at least one face recorded')
mx.flow('blocks carrying street commerce', int(B.street_any.sum()),
        'stalls on at least one recorded face')

print(f'\npopulation on a block whose street commerce is unknown: '
      f'{int(B.loc[~B.street_known, "POBTOT"].sum()):,} '
      f'({B.loc[~B.street_known, "POBTOT"].sum() / B.POBTOT.sum():.2%})')

## 6. The blocks the audit did not reach

Street commerce is unknown, rather than absent, for two kinds of block: those
with no row in the audit, and those with a row on which no face was recorded.
They are counted by federal entity, by metropolitan area and by locality,
together with their population. A block of either kind keeps its place in the
frame as an origin, because its population is enumerated and its distance to
street commerce elsewhere is measurable, and no such block serves as a
destination.

In [ ]:
U = B.loc[~B.street_known].copy()
U['reason'] = np.where(U['in_audit'], 'no face recorded', 'no row in the audit')

print(f'blocks whose street commerce is unknown   {len(U):,} '
      f'({len(U) / len(B):.2%})')
print(f'their population                          '
      f'{int(U.POBTOT.sum()):,} '
      f'({U.POBTOT.sum() / B.POBTOT.sum():.2%} of the frame)')
print()
print(U.groupby('reason').agg(blocks=('CVEGEO_MZA', 'size'),
                              population=('POBTOT', 'sum'),
                              uninhabited=('POBTOT',
                                           lambda s: int((s == 0).sum())))
       .to_string())

print('\n--- by federal entity, the fifteen with the most ---\n')
ent = (U.groupby('CVE_ENT').agg(blocks=('CVEGEO_MZA', 'size'),
                                population=('POBTOT', 'sum'))
         .join(B.groupby('CVE_ENT').agg(blocks_in_frame=('CVEGEO_MZA', 'size'),
                                        pop_in_frame=('POBTOT', 'sum'))))
ent['share_of_blocks'] = (ent.blocks / ent.blocks_in_frame).round(4)
ent['share_of_population'] = (ent.population / ent.pop_in_frame).round(4)
print(ent.sort_values('blocks', ascending=False).head(15).to_string())
rep.add_table('blocks of unknown street commerce by entity',
              ent.reset_index(), max_rows=32)

print('\n--- by area, the fifteen with the largest share of population ---\n')
area = (U.groupby('slug').agg(blocks=('CVEGEO_MZA', 'size'),
                              population=('POBTOT', 'sum'))
          .join(B.groupby('slug').agg(blocks_in_frame=('CVEGEO_MZA', 'size'),
                                      pop_in_frame=('POBTOT', 'sum'))))
area['share_of_blocks'] = (area.blocks / area.blocks_in_frame).round(4)
area['share_of_population'] = (area.population / area.pop_in_frame).round(4)
print(area.sort_values('share_of_population', ascending=False)
          .head(15).to_string())
rep.add_table('blocks of unknown street commerce by area',
              area.reset_index(), max_rows=92)

print('\n--- the fifteen localities holding the most ---\n')
U['CVE_LOC'] = U['CVEGEO_MZA'].str[:9]
loc = (U.groupby(['CVE_LOC', 'slug']).agg(blocks=('CVEGEO_MZA', 'size'),
                                          population=('POBTOT', 'sum'))
         .sort_values('blocks', ascending=False).reset_index())
print(loc.head(15).to_string(index=False))
rep.add_table('localities holding blocks of unknown street commerce', loc,
              max_rows=40)

## 7. The audit against the census

The audit publishes its own count of people and of inhabited private dwellings
for each block, and the census publishes the same quantities for the same
blocks. The two are compared here, which shows whether the join has matched the
same units in both sources.

In [ ]:
chk = B.loc[B.in_audit, ['POBTOT', 'PERSONAS', 'VIVPARHAB']].dropna()
chk['diff'] = chk['PERSONAS'] - chk['POBTOT']

print(f'blocks with a population in both sources  {len(chk):,}')
print(f'population, census                        {chk.POBTOT.sum():,.0f}')
print(f'population, audit                         {chk.PERSONAS.sum():,.0f}')
print(f'blocks agreeing exactly                   '
      f'{int((chk["diff"] == 0).sum()):,} ({(chk["diff"] == 0).mean():.2%})')
print('\ndifference, audit less census:')
print(chk['diff'].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])
       .to_string())

corr = float(chk['POBTOT'].corr(chk['PERSONAS']))
print(f'\ncorrelation between the two counts        {corr:.6f}')
rep.note(f'audit and census block populations correlate at {corr:.6f}')

## 8. Who lives where street commerce is

Where street commerce is matters for the interpretation of every other set of
destinations, because a neighbourhood with stalls on its own blocks has a
source of food that no register records. The blocks and the population carrying
it are reported by deprivation grade, by official classification of the
metropolitan area, and by macro region, for semi fixed stalls on their own and
for either kind together.

In [ ]:
BT = B.merge(TRACTS[['CVEGEO', 'grs_grade', 'veh_share']], on='CVEGEO',
             how='left')

for frame in (B, BT):
    frame['_pop_semi'] = frame['POBTOT'].where(frame['street_semi'], 0.0)
    frame['_pop_ambu'] = frame['POBTOT'].where(frame['street_ambu'], 0.0)
    frame['_pop_any'] = frame['POBTOT'].where(frame['street_any'], 0.0)


def profile(frame, by):
    # Blocks and population carrying street commerce, within each group.
    out = (frame.groupby(by, dropna=False)
                .agg(blocks=('CVEGEO_MZA', 'size'),
                     population=('POBTOT', 'sum'),
                     blocks_semi=('street_semi', 'sum'),
                     blocks_any=('street_any', 'sum'),
                     pop_semi=('_pop_semi', 'sum'),
                     pop_any=('_pop_any', 'sum')))
    out['share_blocks_any'] = (out.blocks_any / out.blocks).round(4)
    out['share_pop_semi'] = (out.pop_semi / out.population).round(4)
    out['share_pop_any'] = (out.pop_any / out.population).round(4)
    return out[['blocks', 'population', 'blocks_semi', 'blocks_any',
                'share_blocks_any', 'share_pop_semi', 'share_pop_any']]

print('--- by deprivation grade ---\n')
grade = profile(BT, 'grs_grade').reindex(mx.GRADE_ORDER + [np.nan])
print(grade.to_string())
rep.add_table('street commerce by deprivation grade',
              grade.reset_index(), max_rows=10)

print('\n--- by macro region ---\n')
region = profile(B, 'macro_region')
print(region.to_string())
rep.add_table('street commerce by macro region', region.reset_index(),
              max_rows=10)

metros = mx.load_metros()
tipo = metros.assign(slug=mx._slug_name(metros['metro_key']).values)[
    ['slug', 'metro_name', 'tipo']]
BM = B.merge(tipo, on='slug', how='left')
assert BM['tipo'].notna().all(), 'a block sits in an area with no classification'

print('\n--- by official classification ---\n')
official = profile(BM, 'tipo').reindex(mx.TIPO_ORDER)
print(official.to_string())
rep.add_table('street commerce by official classification',
              official.reset_index(), max_rows=10)

In [ ]:
by_metro = (profile(BM, 'slug').reset_index()
              .merge(tipo, on='slug')
              .sort_values('share_pop_any', ascending=False))
SHOW = ['metro_name', 'tipo', 'blocks', 'population', 'blocks_any',
        'share_blocks_any', 'share_pop_semi', 'share_pop_any']

print('--- the twenty areas where street commerce reaches the most people ---\n')
print(by_metro.head(20)[SHOW].to_string(index=False))

print('\n--- the twenty where it reaches the fewest ---\n')
print(by_metro.tail(20)[SHOW].to_string(index=False))

rep.add_table('street commerce by area', by_metro, max_rows=92)

## 9. Checks

Each check names the condition under which it fails.

In [ ]:
print('--- checks ---\n')

# A block appearing twice would count its population twice downstream.
d = int(B['CVEGEO_MZA'].duplicated().sum())
print(f'1. duplicate blocks                          {d}')
assert d == 0

# The output has to cover the frame exactly, since every block is an origin.
print(f'2. blocks in the output against the frame    {len(B):,} '
      f'against {len(BLOCKS):,}')
assert len(B) == len(BLOCKS)

# Stalls cannot be recorded on more faces than were recorded for the block.
bad = int(((B['faces_semi'] > B['faces_recorded'])
           | (B['faces_ambu'] > B['faces_recorded'])).sum())
print(f'3. blocks with stalls on more faces than recorded {bad}')
assert bad == 0

# Presence and the count of faces are two expressions of the same thing.
mismatch = int(((B['street_semi'] != (B['faces_semi'] > 0))
                | (B['street_ambu'] != (B['faces_ambu'] > 0))).sum())
print(f'4. presence against the count of faces       {mismatch}')
assert mismatch == 0

# The flag for both kinds has to agree with the two it is built from.
bad = int((B['street_both'] != (B['street_semi'] & B['street_ambu'])).sum())
print(f'5. the both-kinds flag against its parts     {bad}')
assert bad == 0

# A block with stalls recorded must be a block whose street commerce is known.
bad = int((B['street_any'] & ~B['street_known']).sum())
print(f'6. blocks with stalls but unknown coverage   {bad}')
assert bad == 0

# An area with no street commerce at all is possible, so it is listed.
nostreet = sorted(set(B.slug) - set(B.loc[B.street_any, 'slug']))
print(f'7. areas with no street commerce             {len(nostreet)}')
if nostreet:
    print('   ', ', '.join(nostreet))

print(f'\n8. blocks carrying street commerce           '
      f'{int(B.street_any.sum()):,} ({B.street_any.mean():.2%})')
print(f'9. people on a block with street commerce    '
      f'{int(B.loc[B.street_any, "POBTOT"].sum()):,} '
      f'({B.loc[B.street_any, "POBTOT"].sum() / B.POBTOT.sum():.2%})')
print(f'10. share among the blocks where it is known '
      f'{B.loc[B.street_known, "street_any"].mean():.2%}')
print(f'11. blocks absent from the audit             '
      f'{int((~B.in_audit).sum()):,}')
print(f'12. blocks with no face recorded             '
      f'{int((B.in_audit & ~B.street_recorded).sum()):,}')
print(f'13. blocks whose street commerce is unknown  '
      f'{int((~B.street_known).sum()):,} '
      f'({(~B.street_known).mean():.2%})')

## 10. Write results

In [ ]:
keep = ['CVEGEO_MZA', 'CVEGEO', 'CVE_ENT', 'slug', 'macro_region', 'POBTOT',
        'sides_total', 'faces_recorded', 'faces_semi', 'faces_ambu',
        'share_semi', 'share_ambu', 'street_semi', 'street_ambu',
        'street_any', 'street_both', 'in_audit', 'street_recorded',
        'street_known']

out = os.path.join(RESULTS, '06_street_commerce.parquet')
B[keep].to_parquet(out, index=False)
print('written:', out, f'({os.path.getsize(out) / 1e6:.1f} MB)')

print()
print(mx.flow_table().to_string(index=False))
mx.save_flow(os.path.join(mx.DIRS['logs'], '06_sample_flow.csv'))
rep.add_table('attrition', mx.flow_table(), max_rows=20)

rep.note(f'{len(B):,} blocks; {int(B.street_any.sum()):,} carry street '
         f'commerce on at least one recorded face; '
         f'{int(B.loc[B.street_any, "POBTOT"].sum()):,} people live on such a '
         f'block; street commerce is unknown for {int((~B.street_known).sum()):,} '
         f'blocks holding {int(B.loc[~B.street_known, "POBTOT"].sum()):,} people')
rep.save()